# Build a `MarketContext` from raw quotes — canonical path tour

This notebook walks through the analyst-facing flow for building a
`MarketContext` from a JSON `CalibrationEnvelope`. It is the canonical
entry point — `calibrate(envelope_json).market` — and the same surface is
exposed in Rust (`finstack_valuations::calibration::api::engine::execute`)
and JavaScript (`valuations.calibrate(envelopeJson)`).

We load one of the reference envelopes shipped under
`finstack/valuations/examples/market_bootstrap/`, run it through the
calibration engine, verify residuals, and access the resulting context.

In [ ]:
import json
from pathlib import Path

from finstack.valuations import calibrate

# Path to the reference envelope. Adjust if running outside the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "finstack" / "valuations" / "examples" / "market_bootstrap").exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("could not locate the repo root from cwd")
    REPO_ROOT = REPO_ROOT.parent

envelope_path = (
    REPO_ROOT
    / "finstack"
    / "valuations"
    / "examples"
    / "market_bootstrap"
    / "01_usd_discount.json"
)
envelope_json = envelope_path.read_text()
envelope = json.loads(envelope_json)
print(f"schema: {envelope['schema']}")
print(f"plan id: {envelope['plan']['id']}")
print(f"steps: {[step['id'] for step in envelope['plan']['steps']]}")

In [ ]:
result = calibrate(envelope_json)
print(f"success: {result.success}")
print(f"rmse: {result.rmse:.3e}")
print(f"max residual: {result.max_residual:.3e}")
print(f"iterations: {result.iterations}")
result.report_to_dataframe()

In [ ]:
# `result.market` is the calibrated MarketContext, ready for downstream use.
market = result.market

# Look up the discount curve we just built.
curve = market.get_discount("USD-OIS")
print(f"discount factor at t=0:  {curve.df(0.0):.6f}")
print(f"discount factor at t=1y: {curve.df(1.0):.6f}")
print(f"discount factor at t=5y: {curve.df(5.0):.6f}")

## Persisting the materialized market

`result.market_json` returns the same context as a JSON snapshot. This is
useful for caching a calibrated market between processes or for diff'ing
two calibrated states.

`MarketContext` round-trips through this snapshot losslessly:
deserialize via `MarketContext.from_json(...)` (Python) or
`MarketContext::try_from(state)` (Rust).

In [ ]:
# Round-trip via the materialized JSON snapshot.
snapshot_json = result.market_json
print(f"snapshot length: {len(snapshot_json)} bytes")
# Phase 2 of the notebook tour will demonstrate composing markets and
# pulling FX cross rates / bond prices / equity spots from initial_market.

## Compose markets — chained envelope

An analyst's morning bootstrap typically chains discount → hazard → base correlation
in a single envelope. `12_full_credit_desk_market.json` shows the full pattern: rates
and CDS quotes feed two calibration steps, and a tranche quote_set drives the third.
FX cross rates ride along in `initial_market` since they're snapshot-only data.

In [ ]:
envelope_path = REPO_ROOT / "finstack" / "valuations" / "examples" / "market_bootstrap" / "12_full_credit_desk_market.json"
envelope_json = envelope_path.read_text()
result = calibrate(envelope_json)
print(f"success: {result.success}")
print(f"steps: {result.step_ids}")
print(f"rmse: {result.rmse:.3e}")
result.report_to_dataframe()

In [ ]:
import json

market = result.market
discount = market.get_discount("USD-OIS")
hazard = market.get_hazard("CDX-NA-IG-46")

# Base-correlation curves are stored in the market snapshot; reconstruct via the type directly.
from finstack.core.market_data import BaseCorrelationCurve
snapshot = json.loads(market.to_json())
bc_data = next(c for c in snapshot["curves"] if c["type"] == "base_correlation")
bc = BaseCorrelationCurve(id=bc_data["id"], knots=list(zip(bc_data["detachment_points"], bc_data["correlations"])))

print(f"USD-OIS DF(5y):    {discount.df(5.0):.6f}")
print(f"CDX-IG-46 SP(5y):  {hazard.survival(5.0):.6f}")
print(f"BC at 7%:          {bc.correlation(7.0):.4f}")

## Snapshot-only data — FX, bonds, equities

FX matrices, bond prices, equity spots, and dividend schedules are not bootstrapped
today — they ride in `initial_market`. The reference envelopes 09 / 10 / 11
demonstrate each pattern.

In [ ]:
for name in ["09_fx_matrix.json", "10_bond_prices.json", "11_equity_spots_dividends.json"]:
    path = REPO_ROOT / "finstack" / "valuations" / "examples" / "market_bootstrap" / name
    result = calibrate(path.read_text())
    print(f"{name}: success={result.success}, steps={result.step_ids}")

In [ ]:
# Bond prices are snapshot-only and live in the market JSON under the 'prices' key.
bond_envelope = (REPO_ROOT / "finstack" / "valuations" / "examples" / "market_bootstrap" / "10_bond_prices.json").read_text()
bond_market = calibrate(bond_envelope).market
prices = json.loads(bond_market.to_json()).get("prices", {})
for bond_id in ["US-TREASURY-10Y-2026-05-08", "IBM-7YR-2033"]:
    p = prices[bond_id]["price"]
    print(f"{bond_id}: {p['amount']} {p['currency']}")

## Validate before solving

Phase 4 will add `dry_run(envelope_json)` for fast pre-flight validation
(missing dependencies, undefined quote sets, quote class mismatches) — running
structural checks in microseconds before the (much slower) solver. For now,
`validate_calibration_json` returns the canonical pretty-printed JSON and
surfaces serde-level errors; it is the only validation tool until Phase 4 lands.

In [ ]:
from finstack.valuations import validate_calibration_json
envelope_path = REPO_ROOT / "finstack" / "valuations" / "examples" / "market_bootstrap" / "01_usd_discount.json"
canonical = validate_calibration_json(envelope_path.read_text())
print(canonical[:200] + "...")